# Data retrieval: how each source is called

This notebook shows every call the pipeline makes to collect raw data, with a small runnable example of each. It does **not** build features or labels; see `docs/data_retrieval.md` for the full plan and `Sid - EDA.ipynb` for the sample feature table.

| Source | What we pull | Access | Limit |
|---|---|---|---|
| ListenBrainz API | Sitewide top artists, per-artist listener stats, fresh releases, similar artists | `https://api.listenbrainz.org/1/...` | Header-driven, about 30 requests per window |
| MusicBrainz API | Artist metadata, release groups, recordings with credits, releases with labels | `https://musicbrainz.org/ws/2/...` | 1 request/second, descriptive User-Agent required |
| ListenBrainz Spark dump | Every public listen (timestamp, user, artist MBIDs) | `https://data.metabrainz.org/pub/musicbrainz/listenbrainz/fullexport/` | Plain HTTPS with range requests; no database needed |

**Run from the repo root.** Every API response is cached under `data/raw/`, so rerunning a cell makes no new calls. The long dump steps are behind flags (`BUILD_FULL_INDEX`, `RUN_MONTH_SCAN`) that are off by default.

## 1. Setup: sessions, rate limits and the cache

In [ ]:
import csv
import hashlib
import io
import json
import os
import re
import tarfile
import time
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow.compute as pc
import pyarrow.parquet as pq
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

USER_AGENT = "TeamTalentScouts/0.1 ( https://github.com/sidmoor/Predicting-Breakout-Success-of-Emerging-Artists )"
API_CACHE = Path("data/raw/api_cache")
DUMP_CACHE = Path("data/raw/lb_dumps")


def session(retry_statuses):
    """requests.Session with our User-Agent and automatic retries on dropped connections / server errors."""
    s = requests.Session()
    s.headers.update({"User-Agent": USER_AGENT, "Accept": "application/json"})
    s.mount("https://", HTTPAdapter(max_retries=Retry(
        total=5, backoff_factor=1, status_forcelist=retry_statuses, allowed_methods=None)))
    return s


lb = session([500, 502, 503, 504])        # ListenBrainz API (429 handled in lb_get)
mb = session([500, 502, 504])             # MusicBrainz API (503 = rate limit, handled in mb_get)
dl = session([429, 500, 502, 503, 504])   # dump server: long transfers
if os.getenv("LISTENBRAINZ_TOKEN"):       # optional; raises LB rate limits
    lb.headers["Authorization"] = f"Token {os.environ['LISTENBRAINZ_TOKEN']}"


def cached_json(key, fetch):
    """Return fetch() and keep it on disk, so reruns skip API calls that already succeeded."""
    path = API_CACHE / f"{key}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    data = fetch()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data), encoding="utf-8")
    return data

## 2. ListenBrainz API

`lb_get` waits out a `429` for the number of seconds in the `X-RateLimit-Reset-In` header, then retries.

In [ ]:
LB_BASE_URL = "https://api.listenbrainz.org/1"


def lb_get(endpoint, params=None, max_retries=3):
    """GET a ListenBrainz endpoint, backing off on rate limits (429)."""
    url = f"{LB_BASE_URL}/{endpoint.lstrip('/')}"
    for _ in range(max_retries):
        resp = lb.get(url, params=params, timeout=60)
        if resp.status_code == 429:
            time.sleep(int(resp.headers.get("X-RateLimit-Reset-In", 5)))
            continue
        resp.raise_for_status()
        return resp
    resp.raise_for_status()

### 2a. Sitewide top artists
`GET /1/stats/sitewide/artists`: `range` is relative to today (`week`, `month`, `year`, `this_year`, `all_time`, ...), and pages of up to 100 come back with `offset`. We use it for the "top artist" baseline and the listener ceiling that defines "emerging".

In [ ]:
resp = lb_get("stats/sitewide/artists", params={"range": "year", "count": 100, "offset": 0})
print("Rate limit remaining:", resp.headers.get("X-RateLimit-Remaining"), "/", resp.headers.get("X-RateLimit-Limit"))
top_100 = pd.DataFrame(resp.json()["payload"]["artists"])
top_100.head()

### 2b. Per-artist listener stats
`GET /1/stats/artist/{mbid}/listeners?range=year`: total listens, distinct listeners and the top listeners. LB returns no content (`204`) for artists without stats, which is common for small artists. Like every stats endpoint, the range is relative to **today**, so this can't be used for past cohorts.

In [ ]:
def lb_listener_stats(mbid):
    """Last-year listener stats for one artist (cached), or None if LB has none."""
    def fetch():
        resp = lb_get(f"stats/artist/{mbid}/listeners", params={"range": "year"})
        return resp.json()["payload"] if resp.status_code == 200 and resp.content else None
    return cached_json(f"lb_listeners_year/{mbid}", fetch)


example = top_100["artist_mbid"].iloc[0]
stats = lb_listener_stats(example)
{k: stats[k] for k in ("artist_name", "total_listen_count", "total_user_count")}

### 2c. Fresh releases (candidate pool)
`GET /1/explore/fresh-releases/`: releases in the `days` (max 90) before `release_date`, with their artist MBIDs. Calls take 30–60 s, so the responses are cached in `data/raw/lb_api/`; quarterly calls from 2023-04-01 to 2026-01-01 are already there.

In [ ]:
def fresh_releases(release_date):
    cache = Path("data/raw/lb_api") / f"fresh_releases_{release_date:%Y-%m-%d}.json"
    if not cache.exists():
        resp = dl.get(f"{LB_BASE_URL}/explore/fresh-releases/", timeout=180, params={
            "release_date": f"{release_date:%Y-%m-%d}", "days": 90, "past": "true", "future": "false"})
        resp.raise_for_status()
        cache.parent.mkdir(parents=True, exist_ok=True)
        cache.write_text(resp.text, encoding="utf-8")
    return json.loads(cache.read_text(encoding="utf-8"))["payload"]["releases"]


releases = fresh_releases(pd.Timestamp("2025-01-01"))
print(f"{len(releases):,} releases from 2024-10-03 to 2025-01-01")
pd.DataFrame(releases)[["release_date", "artist_credit_name", "release_name", "artist_mbids"]].head()

### 2d. Similar artists (LB Labs)
Similar artists computed from LB listening sessions. This is a **current snapshot**, so it can't be rebuilt for past dates (see the issues section of the plan).

In [ ]:
SIMILAR_ALGORITHM = "session_based_days_7500_session_300_contribution_5_threshold_10_limit_100_filter_True_skip_30"


def similar_artists(mbid):
    """Top 10 similar artists from LB Labs, cached."""
    def fetch():
        resp = lb.get("https://labs.api.listenbrainz.org/similar-artists/json",
                      params={"artist_mbids": mbid, "algorithm": SIMILAR_ALGORITHM}, timeout=60)
        return [a["artist_mbid"] for a in resp.json()[:10]] if resp.ok else []
    return cached_json(f"lb_similar/{mbid}", fetch)


similar_artists(example)[:5]

## 3. MusicBrainz API

`mb_get` keeps calls at least 1.1 s apart and retries on `503`, which is how MusicBrainz signals a rate limit. Browse endpoints return 100 items per page; `mb_browse` pages through them up to a cap.

In [ ]:
MB_BASE_URL = "https://musicbrainz.org/ws/2"
_mb_last_call = 0.0


def mb_get(endpoint, params=None, max_retries=3):
    """GET a MusicBrainz ws/2 endpoint (cached), spacing calls 1.1 s apart and retrying on 503."""
    params = {**(params or {}), "fmt": "json"}

    def fetch():
        global _mb_last_call
        for _ in range(max_retries):
            time.sleep(max(0.0, 1.1 - (time.time() - _mb_last_call)))
            _mb_last_call = time.time()
            resp = mb.get(f"{MB_BASE_URL}/{endpoint}", params=params, timeout=30)
            if resp.status_code == 503:
                continue
            resp.raise_for_status()
            return resp.json()
        resp.raise_for_status()

    key = hashlib.sha1(f"{endpoint}?{sorted(params.items())}".encode()).hexdigest()
    return cached_json(f"mb/{key}", fetch)


def mb_browse(entity, params, max_pages):
    """Page through an MB browse endpoint (100 items per page), stopping after max_pages."""
    items = []
    for page in range(max_pages):
        data = mb_get(entity, {**params, "limit": 100, "offset": page * 100})
        items += data[f"{entity}s"]
        if len(items) >= data[f"{entity}-count"]:
            break
    return items

The four calls per artist, shown for one emerging artist from the sample:

| Call | Used for |
|---|---|
| `artist/{mbid}?inc=genres+tags` | Type, country, area, genres |
| `release-group?artist={mbid}` | First release date, release counts and cadence |
| `recording?artist={mbid}&inc=artist-credits` | Collaborators (co-credited artists) |
| `release?artist={mbid}&inc=labels` | Record labels |

In [ ]:
mbid = "e320beac-a7f9-4ad2-a786-fcb5e1f38608"  # an emerging artist from data/processed/sample_features.csv

artist = mb_get(f"artist/{mbid}", {"inc": "genres+tags"})
release_groups = mb_browse("release-group", {"artist": mbid}, max_pages=3)
recordings = mb_browse("recording", {"artist": mbid, "inc": "artist-credits"}, max_pages=5)
releases_mb = mb_browse("release", {"artist": mbid, "inc": "labels"}, max_pages=3)

print(artist["name"], "|", artist.get("type"), "|", artist.get("country"))
print(f"{len(release_groups)} release groups, {len(recordings)} recordings, {len(releases_mb)} releases")
pd.DataFrame([{"title": r["title"], "type": r.get("primary-type"), "first_release_date": r.get("first-release-date")}
              for r in release_groups]).sort_values("first_release_date")

## 4. ListenBrainz Spark dump (listening history)

The **full export** is a complete snapshot of every public listen since ListenBrainz began; the date in its name is when the snapshot was taken, not how far back it goes. The *Spark* variant is an **uncompressed `.tar` of Parquet files** in time order (each file covers ~3–5 days). No database or special software is needed:

1. Walk the tar's 512-byte headers with range requests to get each file's byte offset (the **index**, ~1 h once per dump, cached).
2. Open any Parquet file inside the tar as a seekable remote file (`HTTPRangeFile`) and read only the 3 columns we need.
3. Aggregate with DuckDB and keep only the small monthly tables.

Full dumps are replaced every couple of weeks and old ones are deleted, so the index has to be rebuilt for each new dump.

In [ ]:
DUMP_ROOT = "https://data.metabrainz.org/pub/musicbrainz/listenbrainz/fullexport/"
NEEDED_COLS = ["listened_at", "user_id", "artist_credit_mbids"]

listing = dl.get(DUMP_ROOT, timeout=30).text
dump_dirs = sorted(set(re.findall(r'href="(listenbrainz-dump-(\d+)-\d{8}-\d+-full)/"', listing)), key=lambda d: int(d[1]))
DUMP_DIR, DUMP_ID = dump_dirs[-1]
dump_files = re.findall(r'href="([^"]+\.tar(?:\.zst)?)"', dl.get(DUMP_ROOT + DUMP_DIR + "/", timeout=30).text)
SPARK_URL = f"{DUMP_ROOT}{DUMP_DIR}/" + next(f for f in dump_files if "spark-dump" in f)
print("Latest full dump:", DUMP_DIR)
print("Spark dump size:", round(int(dl.head(SPARK_URL, timeout=30).headers["Content-Length"]) / 1e9, 1), "GB")

### 4a. Reading the tar without downloading it

In [ ]:
class HTTPRangeFile(io.RawIOBase):
    """Seekable read-only view of bytes [start, start + size) of a remote file, fetched with HTTP Range requests."""

    def __init__(self, url, start, size, session=dl):
        self.url, self.start, self.size, self.session = url, start, size, session
        self.pos = 0
        self.bytes_fetched = 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=io.SEEK_SET):
        self.pos = {io.SEEK_SET: offset, io.SEEK_CUR: self.pos + offset, io.SEEK_END: self.size + offset}[whence]
        return self.pos

    def readinto(self, buf):
        n = min(len(buf), self.size - self.pos)
        if n <= 0:
            return 0
        first = self.start + self.pos
        resp = self.session.get(self.url, headers={"Range": f"bytes={first}-{first + n - 1}"}, timeout=120)
        resp.raise_for_status()
        buf[:n] = resp.content
        self.pos += n
        self.bytes_fetched += n
        return n


def iter_tar_members(url, offset=0):
    """Yield (name, data_offset, size) for each file in a remote uncompressed tar, one ~2 s request per member."""
    while True:
        buf = dl.get(url, headers={"Range": f"bytes={offset}-{offset + 2047}"}, timeout=60).content
        pos = 0
        while pos + 512 <= len(buf):
            block = buf[pos:pos + 512]
            if block == b"\0" * 512:  # a zero block marks the end of a tar
                return
            info = tarfile.TarInfo.frombuf(block, "utf-8", "surrogateescape")
            if info.type == tarfile.XHDTYPE:  # pax header: skip it and its data
                pos += 512 + (info.size + 511) // 512 * 512
                continue
            data_offset = offset + pos + 512
            yield info.name.split("/")[-1], data_offset, info.size
            offset = data_offset + (info.size + 511) // 512 * 512
            break
        else:
            offset += pos


def build_tar_index(url, cache_path):
    """Full index of the tar as a DataFrame. Rows are appended as we go, so an interrupted run resumes."""
    if not (cache_path.exists() and cache_path.read_text().rstrip().endswith("#complete")):
        offset = 0
        if cache_path.exists() and len(cache_path.read_text().splitlines()) > 1:
            name, data_offset, size = cache_path.read_text().splitlines()[-1].split(",")
            offset = int(data_offset) + (int(size) + 511) // 512 * 512
        else:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            cache_path.write_text("name,data_offset,size\n")
        with cache_path.open("a", newline="") as f:
            writer = csv.writer(f)
            for row in iter_tar_members(url, offset):
                writer.writerow(row)
                f.flush()
            f.write("#complete\n")
    return pd.read_csv(cache_path, comment="#")

### 4b. Example: read one Parquet file from the dump
Walks headers only until the first Parquet file (a few requests), then reads its 3 needed columns. This is exactly what the full pass does for every file in the window. The first file is tiny (about 1,100 listens from October 2002); a typical 2021+ file is ~180 MB with ~2 M listens, of which we download ~15%.

In [ ]:
for name, data_offset, size in iter_tar_members(SPARK_URL):
    if name.endswith(".parquet"):
        break

t0 = time.time()
member = HTTPRangeFile(SPARK_URL, data_offset, size)
listens = pq.ParquetFile(member).read(columns=NEEDED_COLS)
print(f"{name}: {listens.num_rows:,} listens; downloaded {member.bytes_fetched / 1e6:.1f} of {size / 1e6:.1f} MB "
      f"in {time.time() - t0:.0f}s")
print("Time range:", pc.min(listens["listened_at"]).as_py(), "to", pc.max(listens["listened_at"]).as_py())
listens.slice(0, 5).to_pandas()

### 4c. Aggregate to monthly tables
Per (artist, month): listens, unique listeners and repeat listeners (users with 2+ listens); per month: active users across the whole platform, used to normalize for LB's own growth. Listens with no artist MBID (~14%) are dropped.

A month spans ~8–10 files, so the full pass saves per-file `(month, user, artist)` counts and re-aggregates once all of a month's files are in; unique listeners can't be summed across files.

In [ ]:
AGG_SQL = """
    WITH per_user AS (
        SELECT artist_mbid, month, user_id, count(*) AS n
        FROM (SELECT unnest(artist_credit_mbids) AS artist_mbid, date_trunc('month', listened_at) AS month, user_id
              FROM listens WHERE len(artist_credit_mbids) > 0)
        GROUP BY ALL
    )
    SELECT artist_mbid, month, sum(n) AS listens, count(*) AS unique_listeners,
           count(*) FILTER (n >= 2) AS repeat_listeners
    FROM per_user GROUP BY ALL
"""
con = duckdb.connect()
con.register("listens", listens)
artist_month_example = con.sql(AGG_SQL).df()
platform_month_example = con.sql("""
    SELECT date_trunc('month', listened_at) AS month, count(DISTINCT user_id) AS active_users, count(*) AS listens,
           avg((len(artist_credit_mbids) > 0)::INT) AS mapped_share
    FROM listens GROUP BY ALL ORDER BY month
""").df()
print(platform_month_example)
artist_month_example.sort_values("unique_listeners", ascending=False).head()

### 4d. Full pass (off by default)
With the index built, `first_file_on_or_after` finds a month's files by binary search over `listened_at`, and `scan_month` reads them in parallel and writes that month's tables. Set the flags to `True` to run: the index takes ~1 h per dump, and each month ~1–2 min with 4 parallel downloads.

In [ ]:
BUILD_FULL_INDEX = False
RUN_MONTH_SCAN = False      # needs the index
SCAN_MONTH = pd.Timestamp("2024-12-01")

from concurrent.futures import ThreadPoolExecutor


def first_file_on_or_after(parquet_index, ts):
    """Index of the first parquet file containing listens on/after ts (binary search; files are in time order)."""
    def max_listened_at(i):
        row = parquet_index.iloc[i]
        col = pq.ParquetFile(HTTPRangeFile(SPARK_URL, int(row["data_offset"]), int(row["size"]))).read(
            columns=["listened_at"])["listened_at"]
        return pd.Timestamp(pc.max(col).as_py())

    lo, hi = 0, len(parquet_index) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if max_listened_at(mid) < ts:
            lo = mid + 1
        else:
            hi = mid
    return lo


def scan_month(parquet_index, month, out_dir=Path("data/interim/artist_month")):
    """Write artist_month and platform_month parquet files for one calendar month."""
    end = month + pd.offsets.MonthBegin(1)
    ids = range(first_file_on_or_after(parquet_index, month), first_file_on_or_after(parquet_index, end) + 1)
    in_month = f"listened_at >= TIMESTAMP '{month}' AND listened_at < TIMESTAMP '{end}'"

    def read(i):
        row = parquet_index.iloc[i]
        return pq.ParquetFile(HTTPRangeFile(SPARK_URL, int(row["data_offset"]), int(row["size"]))).read(columns=NEEDED_COLS)

    con = duckdb.connect()
    con.execute("CREATE TABLE per_user (artist_mbid VARCHAR, user_id BIGINT, n BIGINT)")
    con.execute("CREATE TABLE platform (user_id BIGINT, n BIGINT)")
    with ThreadPoolExecutor(max_workers=4) as pool:
        for table in pool.map(read, ids):
            con.register("listens", table)
            con.execute(f"""INSERT INTO per_user SELECT artist_mbid, user_id, count(*) FROM (
                SELECT unnest(artist_credit_mbids) AS artist_mbid, user_id FROM listens
                WHERE {in_month} AND len(artist_credit_mbids) > 0) GROUP BY ALL""")
            con.execute(f"INSERT INTO platform SELECT user_id, count(*) FROM listens WHERE {in_month} GROUP BY ALL")
            con.unregister("listens")
    out_dir.mkdir(parents=True, exist_ok=True)
    con.sql(f"""SELECT artist_mbid, DATE '{month.date()}' AS month, sum(n) AS listens, count(*) AS unique_listeners,
                       count(*) FILTER (n >= 2) AS repeat_listeners
                FROM (SELECT artist_mbid, user_id, sum(n) AS n FROM per_user GROUP BY ALL) GROUP BY ALL
             """).write_parquet(str(out_dir / f"artist_month_{month:%Y-%m}.parquet"))
    con.sql(f"""SELECT DATE '{month.date()}' AS month, count(DISTINCT user_id) AS active_users, sum(n) AS listens
                FROM platform""").write_parquet(str(out_dir / f"platform_month_{month:%Y-%m}.parquet"))
    con.close()


if BUILD_FULL_INDEX or RUN_MONTH_SCAN:
    tar_index = build_tar_index(SPARK_URL, DUMP_CACHE / f"spark_index_{DUMP_ID}.csv")
    parquet_index = tar_index[tar_index["name"].str.endswith(".parquet")].copy()
    parquet_index["file_no"] = parquet_index["name"].str.removesuffix(".parquet").astype(int)
    parquet_index = parquet_index.sort_values("file_no").reset_index(drop=True)
    print(f"{len(parquet_index):,} parquet files indexed")
    if RUN_MONTH_SCAN:
        scan_month(parquet_index, SCAN_MONTH)